# Day 3 · Exercise 2: The OpenAI-Compatible Endpoint

**What you'll build:** `ask_compat` — call the same local model using the `openai` Python package pointed at Ollama's compatible endpoint.

**Why it matters:** The OpenAI-compatible API shape is the industry standard. This exact pattern — change only `base_url` — also works with Groq, Together AI, Mistral, and real OpenAI.

**Prereq check:**
1. Ollama running with `llama3.2` pulled (same as Exercise 1)
2. `pip install openai` in your environment

## Your Implementation

In [ ]:
from openai import OpenAI

MODEL = "llama3.2"

def ask_compat(question: str) -> str:
    """Call the local model via the OpenAI-compatible endpoint.

    Create an OpenAI client pointing at http://localhost:11434/v1 and
    call chat.completions.create() with model and messages.

    Args:
        question: The question to ask.

    Returns:
        The model's text reply as a string.

    Example:
        ask_compat("What is 2 + 2?")  ->  "4"
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

def _ollama_running():
    try:
        import urllib.request  # stdlib — no install needed
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        return True
    except Exception:
        return False

def _run_checks():
    score, total = 0, 4

    # Check 1: function exists and is callable
    try:
        assert callable(ask_compat), 'ask_compat is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: Ollama running + function returns a string
    if not _ollama_running():
        print(f'{_FAIL} Check 2/{total}: Ollama server is not running')
        print('  → macOS: open the Ollama app · Linux/Windows: ollama serve')
        return

    result = None
    try:
        result = ask_compat('Reply with only the word HELLO.')
        assert isinstance(result, str), f'expected str, got {type(result).__name__}'
        print(f'{_PASS} Check 2/{total}: returns a string')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: call failed — {e}')
        print('  → Is base_url set to http://localhost:11434/v1 ?')
        print('  → Is the openai package installed?  pip install openai')
        return

    if result is None:
        return

    # Check 3: result is non-empty
    try:
        assert len(result) > 0, 'returned an empty string'
        print(f'{_PASS} Check 3/{total}: result is non-empty')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: works with a different question
    try:
        r2 = ask_compat('What colour is the sky? One word.')
        assert isinstance(r2, str) and len(r2) > 0
        print(f'{_PASS} Check 4/{total}: works with a different question')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    print()
    if score == total:
        print('=' * 52)
        print(f'  {_PASS}  Exercise 2 complete! {total}/{total} checks passed.')
        print('=' * 52)
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

The openai-compatible pattern works with other free providers too. Try pointing it at **Groq** (fast free cloud inference):
```python
# 1. Sign up free at https://console.groq.com and get an API key
# 2. pip install groq  (or just use openai with base_url)
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="your-groq-key",
)
# model name changes — try: "llama-3.1-8b-instant"
```
Same code, different server, different speed. That's provider portability.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
from openai import OpenAI

MODEL = "llama3.2"

def ask_compat(question: str) -> str:
    client = OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama",  # required by the package, ignored by Ollama
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content
```

**Why this works:** The `openai` package sends a POST to `{base_url}/chat/completions`. By setting `base_url` to your local Ollama server, the request goes to `http://localhost:11434/v1/chat/completions` instead of OpenAI's cloud. Ollama accepts the same JSON format and returns the same response shape, so `response.choices[0].message.content` works exactly as it would with real OpenAI.
</details>